# Model Refinement and Optimization

After developing a baseline regression model, the next step is to enhance model performance by capturing more complex relationships within the data.

In real-world scenarios, relationships between predictors and the target variable are rarely purely linear. Therefore, we extend the baseline model using advanced techniques that allow the model to better represent underlying patterns.

This notebook focuses on improving predictive performance through:

- Polynomial transformations to capture non-linear relationships
- Interaction terms to model dependencies between variables
- Regularization techniques to reduce overfitting and improve generalization

Additionally, multiple models are evaluated and compared using performance metrics such as R-squared and RMSE to identify the most effective modeling approach.

## Import Required Libraries

In this section, we import all necessary libraries required for data handling, model building, transformation, and evaluation.

These libraries enable us to:

- Manipulate and process data
- Generate polynomial and interaction features
- Train regression models
- Evaluate model performance

In [ ]:
# Import pandas for handling structured datasets (DataFrames)
import pandas as pd

# Import numpy for numerical operations such as arrays and mathematical functions
import numpy as np

# Import train_test_split to divide dataset into training and testing sets
from sklearn.model_selection import train_test_split

# Import PolynomialFeatures to create polynomial and interaction features
from sklearn.preprocessing import PolynomialFeatures

# Import regression models: basic linear regression and regularized models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Import evaluation metrics to measure model performance
from sklearn.metrics import mean_squared_error, r2_score

## Load Dataset and Prepare Variables

We load the cleaned dataset generated from the ETL stage. This dataset contains all relevant features required for modeling.

We then separate:

- Target variable: `rented_bike_count`
- Predictor variables: all remaining features

Finally, we split the dataset into training and testing subsets to evaluate model performance on unseen data.

In [ ]:
# Load the cleaned dataset from CSV file into a pandas DataFrame
df = pd.read_csv("seoul_bike_sharing_converted_normalized.csv")

# Define the feature matrix (X) by dropping the target column
X = df.drop(columns=['rented_bike_count'])

# Define the target variable (y) as rented bike count
y = df['rented_bike_count']

# Split the dataset into training (80%) and testing (20%) sets
# random_state ensures reproducibility of results
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Polynomial Regression

Polynomial regression extends linear regression by adding higher-order terms of the predictors.

This allows the model to capture non-linear relationships between variables.

For example:
- Temperature may have a non-linear effect on demand
- Extremely high or low values may reduce usage

We generate polynomial features and train a regression model on the transformed dataset.

In [ ]:
# Create a PolynomialFeatures object to generate polynomial terms up to degree 2
# include_bias=False avoids adding an unnecessary constant column
poly = PolynomialFeatures(degree=2, include_bias=False)

# Fit the polynomial transformer on training data and transform it
X_train_poly = poly.fit_transform(X_train)

# Apply the same transformation to the test dataset
X_test_poly = poly.transform(X_test)

# Initialize a linear regression model
poly_model = LinearRegression()

# Train the model using polynomial features
poly_model.fit(X_train_poly, y_train)

# Generate predictions on the test dataset
y_pred_poly = poly_model.predict(X_test_poly)

In [ ]:
# Calculate R-squared value to measure explained variance
r2_poly = r2_score(y_test, y_pred_poly)

# Calculate RMSE to measure prediction error magnitude
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))

# Print evaluation results
print("Polynomial R2:", r2_poly)
print("Polynomial RMSE:", rmse_poly)

## Interaction Effects

Interaction terms capture relationships where the effect of one feature depends on another.

For example:
- The effect of temperature may depend on humidity
- Rainfall combined with temperature may influence demand differently

We generate interaction-only features to isolate these effects.

In [ ]:
# Create PolynomialFeatures object to generate only interaction terms
# interaction_only=True ensures no squared terms are included
poly_interaction = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# Fit and transform training data
X_train_int = poly_interaction.fit_transform(X_train)

# Transform test data using same mapping
X_test_int = poly_interaction.transform(X_test)

# Initialize regression model
interaction_model = LinearRegression()

# Train model using interaction features
interaction_model.fit(X_train_int, y_train)

# Predict on test dataset
y_pred_int = interaction_model.predict(X_test_int)

In [ ]:
# Compute R-squared for interaction model
r2_int = r2_score(y_test, y_pred_int)

# Compute RMSE for interaction model
rmse_int = np.sqrt(mean_squared_error(y_test, y_pred_int))

# Display results
print("Interaction R2:", r2_int)
print("Interaction RMSE:", rmse_int)

## Regularization Techniques

Regularization helps prevent overfitting by penalizing large model coefficients.

We evaluate three techniques:

- Ridge Regression (L2 penalty)
- Lasso Regression (L1 penalty)
- Elastic Net (combination of L1 and L2)

These methods improve generalization and model stability.

In [ ]:
# Initialize Ridge regression with alpha controlling regularization strength
ridge = Ridge(alpha=1.0)

# Train Ridge model
ridge.fit(X_train, y_train)

# Predict using Ridge model
y_pred_ridge = ridge.predict(X_test)

In [ ]:
# Initialize Lasso regression model
lasso = Lasso(alpha=0.1)

# Train Lasso model
lasso.fit(X_train, y_train)

# Generate predictions
y_pred_lasso = lasso.predict(X_test)

In [ ]:
# Initialize Elastic Net model with balance between L1 and L2 penalties
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)

# Train Elastic Net model
elastic.fit(X_train, y_train)

# Predict using Elastic Net model
y_pred_elastic = elastic.predict(X_test)

In [ ]:
# Define a helper function to compute evaluation metrics
def evaluate(y_true, y_pred):
    
    # Calculate R-squared
    r2 = r2_score(y_true, y_pred)
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Return both metrics
    return r2, rmse

# Store results for each model
results = {
    "Ridge": evaluate(y_test, y_pred_ridge),
    "Lasso": evaluate(y_test, y_pred_lasso),
    "ElasticNet": evaluate(y_test, y_pred_elastic)
}

# Display results dictionary
results

## Model Comparison

To determine the best-performing model, we compare all approaches based on:

- R-squared (higher is better)
- RMSE (lower is better)

This allows us to identify the model that best balances accuracy and generalization.

In [ ]:
# Create a DataFrame summarizing all model performances
comparison = pd.DataFrame({
    
    # List model names
    "Model": ["Polynomial", "Interaction", "Ridge", "Lasso", "ElasticNet"],
    
    # Store R2 values
    "R2": [
        r2_poly,
        r2_int,
        results["Ridge"][0],
        results["Lasso"][0],
        results["ElasticNet"][0]
    ],
    
    # Store RMSE values
    "RMSE": [
        rmse_poly,
        rmse_int,
        results["Ridge"][1],
        results["Lasso"][1],
        results["ElasticNet"][1]
    ]
})

# Display comparison table
comparison

## Interpretation

From the comparison:

- Polynomial models capture non-linearity but may increase complexity
- Interaction models reveal dependencies between variables
- Regularization improves stability and reduces overfitting

Elastic Net often provides the best balance between flexibility and generalization.

This model can be used as the final candidate for deployment in the dashboard.

---

## Author & Acknowledgment

**Author:**  
Deepan Mehta  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on improving regression model performance through advanced techniques including polynomial transformations, interaction effects, and regularization methods.

The structure and methodology are inspired by IBM Skills Network labs on regression model refinement.

Special acknowledgment is given to:

- Yan Luo  
- Jeff Grossman  

---

## Project Context

This notebook represents the model optimization stage in the pipeline:

- Data Collection  
- Data Wrangling  
- EDA  
- Baseline Modeling  
- Model Refinement  

---

## Notes

All code and explanations have been independently rewritten and enhanced for clarity, reproducibility, and real-world applicability.

---